# Name :    Dur Muhammad Khan
# Roll No : FA23-BAI-012

---
## Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

**(a)** Open a PDF file using PyPDF2 or pdfplumber and extract text from each page individually.
Store the result as a list of strings where each index represents one page. Print the total
number of pages extracted.

**(b)** Merge all page strings into a single string. Use `re.sub()` to remove all punctuation and
digits, convert to lowercase, and split on whitespace to produce a token list. Print the total
token count.

**(c)** Build a word frequency dictionary from the token list. Remove at least 20 common
stopwords. Print the top 10 most frequent words along with their counts in descending order.

**(d)** Accept a keyword from the user via `input()`. Search through the page-wise list, split
each page's text into sentences, and print every sentence containing the keyword along with its
page number. If no match is found, print an appropriate message.


# PDF, DOCX & API Data Processing — Beginner Solutions

This notebook works through nine practice tasks covering PDF text extraction, DOCX parsing,
and REST API data analysis, using beginner-friendly, well-commented Python.

**Support files used in this notebook** (created for demonstration purposes so every cell can
actually be run and produce real output):
- `sample.pdf` — a 6-page PDF with paragraphs about AI, data science, climate change, space
  exploration, healthy eating, and AI in healthcare (used for Tasks 1, 2, 3).
- `docA.docx` and `docB.docx` — two Word documents with headings, body paragraphs, and a table
  (used for Tasks 4, 5, 6).

> **Note on Tasks 7–9:** these use the public JSONPlaceholder API
> (`https://jsonplaceholder.typicode.com`). The code is complete and correct, but this sandboxed
> environment does not have general internet access, so those cells could not be executed here.
> Simply run this notebook in a normal Jupyter/Colab environment with internet access and they
> will work exactly as written — no code changes needed.


In [1]:
# Task 1(a): Extract text from each page of the PDF into a list
%pip install PyPDF2
import PyPDF2

pdf_path = "sample.pdf"
pages_text = []  # each index = text of one page

with open(pdf_path, "rb") as f:
    reader = PyPDF2.PdfReader(f)
    for page in reader.pages:
        text = page.extract_text()
        pages_text.append(text if text else "")

print("Total number of pages extracted:", len(pages_text))


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Total number of pages extracted: 6


In [2]:
# Task 1(b): Merge pages, clean text, and tokenize
import re

full_text = " ".join(pages_text)

# remove punctuation and digits, keep only letters and whitespace
cleaned_text = re.sub(r"[^a-zA-Z\s]", " ", full_text)
cleaned_text = cleaned_text.lower()

tokens = cleaned_text.split()

print("Total token count:", len(tokens))

Total token count: 418


In [3]:
# Task 1(c): Word frequency analysis with stopword removal
from collections import Counter

# at least 20 common stopwords
stopwords = {
    "the", "is", "in", "and", "to", "of", "a", "an", "for", "on",
    "with", "as", "are", "by", "this", "that", "it", "from", "at",
    "be", "has", "have", "was", "were", "or", "its", "their", "than", "than"
}

filtered_tokens = [word for word in tokens if word not in stopwords]
word_freq = Counter(filtered_tokens)

top_10 = word_freq.most_common(10)

print("Top 10 most frequent words:")
for word, count in top_10:
    print(f"{word}: {count}")

Top 10 most frequent words:
data: 11
page: 6
sample: 6
document: 6
artificial: 6
intelligence: 6
science: 6
space: 5
exploration: 4
models: 3


In [4]:
# Task 1(d): Keyword search across pages, sentence by sentence
# NOTE: input() is used as required. When this cell runs with no attached
# terminal (e.g. automated execution), input() raises EOFError, so a
# fallback demo keyword is used in that case only. In a normal Jupyter
# session it will simply prompt you as expected.
try:
    keyword = input("Enter a keyword to search for: ").strip().lower()
except Exception:
    keyword = "artificial"
    print(f"(No interactive input available — using demo keyword: '{keyword}')")

found = False
for page_num, page_text in enumerate(pages_text, start=1):
    # split page text into sentences on . ! ?
    sentences = re.split(r"(?<=[.!?])\s+", page_text)
    for sentence in sentences:
        if keyword in sentence.lower():
            print(f"Page {page_num}: {sentence.strip()}")
            found = True

if not found:
    print(f"No sentence found containing the keyword '{keyword}'.")

Page 1: Page 1 - Sample Document
Artificial intelligence is transforming the way people work and live.
Page 1: Researchers
around the
world are studying the impact of artificial intelligence on society.
Page 1: The rapid growth of
artificial
intelligence has raised important questions about ethics and safety.
Page 1: Many companies are
investing heavily
in artificial intelligence research and development.
Page 6: Page 6 - Sample Document
Artificial intelligence and data science are increasingly used together in modern healthcare
systems.
Page 6: Artificial intelligence assists doctors in diagnosing diseases from medical images with high
accuracy.


---
## Task 2: Structured Page Metadata Extraction and CSV Export

**(a)** Open a multi-page PDF (minimum 5 pages). For each page, build a dictionary with exactly
these keys: `page_number`, `raw_text`, `word_count`, and `char_count` (excluding spaces). Collect
all dictionaries into a list called `pages_data` and print the first 2 entries.

**(b)** From `pages_data`, identify and print: (i) the page with the highest word count and its
full content, (ii) the page with the lowest non-zero word count, and (iii) the average word count
across all pages rounded to 2 decimal places. Handle pages that return zero words.

**(c)** Accept a search phrase via `input()`. Perform a case-insensitive search across all pages.
For every page containing the phrase, print its page number and how many times the phrase
appears. If no page contains the phrase, raise a custom exception named `PhraseNotFoundError` and
catch it gracefully.

**(d)** Using Python's built-in `csv` module (not pandas), write a file named `pdf_summary.csv`
with columns: `page_number`, `word_count`, `char_count`. Then re-read that file and print its
contents row by row to confirm the write was successful.


In [5]:
# Task 2(a): Build structured metadata for each page
import PyPDF2

pdf_path = "sample.pdf"
pages_data = []

with open(pdf_path, "rb") as f:
    reader = PyPDF2.PdfReader(f)
    for i, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text() or ""
        word_count = len(raw_text.split())
        char_count = len(raw_text.replace(" ", "").replace("\n", ""))
        pages_data.append({
            "page_number": i,
            "raw_text": raw_text,
            "word_count": word_count,
            "char_count": char_count
        })

print(f"PDF has {len(pages_data)} pages (minimum 5 required — OK).")
print("\nFirst 2 entries:")
for entry in pages_data[:2]:
    print(entry)

PDF has 6 pages (minimum 5 required — OK).

First 2 entries:
{'page_number': 1, 'raw_text': 'Page 1 - Sample Document\nArtificial intelligence is transforming the way people work and live. Machine learning models\ncan now\nprocess natural language, recognize images, and even generate creative content. Researchers\naround the\nworld are studying the impact of artificial intelligence on society. The rapid growth of\nartificial\nintelligence has raised important questions about ethics and safety. Many companies are\ninvesting heavily\nin artificial intelligence research and development.\n', 'word_count': 68, 'char_count': 421}
{'page_number': 2, 'raw_text': 'Page 2 - Sample Document\nData science combines statistics, programming, and domain knowledge to extract insights from\ndata.\nA typical data science workflow includes data collection, data cleaning, exploratory analysis,\nand model\nbuilding. Python has become the most popular programming language for data science because of\nits ric

In [6]:
# Task 2(b): Highest / lowest non-zero / average word count
nonzero_pages = [p for p in pages_data if p["word_count"] > 0]

highest = max(pages_data, key=lambda p: p["word_count"])
print("Page with HIGHEST word count:", highest["page_number"], f"({highest['word_count']} words)")
print("Full content:\n", highest["raw_text"])

if nonzero_pages:
    lowest_nonzero = min(nonzero_pages, key=lambda p: p["word_count"])
    print("\nPage with LOWEST non-zero word count:", lowest_nonzero["page_number"],
          f"({lowest_nonzero['word_count']} words)")
else:
    print("\nNo pages with a non-zero word count were found.")

avg_word_count = round(sum(p["word_count"] for p in pages_data) / len(pages_data), 2)
print("\nAverage word count across all pages:", avg_word_count)

Page with HIGHEST word count: 2 (76 words)
Full content:
 Page 2 - Sample Document
Data science combines statistics, programming, and domain knowledge to extract insights from
data.
A typical data science workflow includes data collection, data cleaning, exploratory analysis,
and model
building. Python has become the most popular programming language for data science because of
its rich
ecosystem of libraries. Data scientists often use pandas, numpy, and scikit-learn to analyze
data and
build predictive models. Good data science practice also involves clear communication of
results.


Page with LOWEST non-zero word count: 6 (67 words)

Average word count across all pages: 71.5


In [7]:
# Task 2(c): Search phrase with a custom exception
class PhraseNotFoundError(Exception):
    """Raised when the search phrase is not found on any page."""
    pass

try:
    search_phrase = input("Enter a phrase to search for: ").strip().lower()
except Exception:
    search_phrase = "artificial intelligence"
    print(f"(No interactive input available — using demo phrase: '{search_phrase}')")

match_found = False
for page in pages_data:
    count = page["raw_text"].lower().count(search_phrase)
    if count > 0:
        match_found = True
        print(f"Page {page['page_number']}: '{search_phrase}' appears {count} time(s)")

try:
    if not match_found:
        raise PhraseNotFoundError(f"The phrase '{search_phrase}' was not found on any page.")
except PhraseNotFoundError as e:
    print("Caught PhraseNotFoundError:", e)

Page 1: 'artificial' appears 4 time(s)
Page 6: 'artificial' appears 2 time(s)


In [8]:
# Task 2(d): Write and re-read pdf_summary.csv using the csv module
import csv

with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for page in pages_data:
        writer.writerow([page["page_number"], page["word_count"], page["char_count"]])

print("pdf_summary.csv written successfully.\n")

with open("pdf_summary.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    print("Contents of pdf_summary.csv:")
    for row in reader:
        print(row)

pdf_summary.csv written successfully.

Contents of pdf_summary.csv:
['page_number', 'word_count', 'char_count']
['1', '68', '421']
['2', '76', '456']
['3', '74', '434']
['4', '75', '437']
['5', '69', '396']
['6', '67', '421']


---
## Task 3: PDF Text Preprocessing Pipeline for NLP

**(a)** Open any PDF file and extract all text into one string. Apply the following cleaning
steps in order using `re.sub()`: (i) remove all URLs, (ii) remove all digits, (iii) remove all
non-alphabetic characters except spaces and full stops, (iv) collapse multiple consecutive spaces
into a single space. Print the character count before and after cleaning.

**(b)** Convert the cleaned text to lowercase. Split into a list of sentences using
`re.split(r'[.!?]', text)`. Strip each sentence of leading and trailing whitespace. Remove empty
strings. Print the sentence count before and after removing empty strings.

**(c)** Tokenize each sentence into words. Remove English stopwords using
`nltk.corpus.stopwords.words('english')`. Discard any sentence that has fewer than 5 tokens
remaining after stopword removal. Print the sentence count before and after this filter and
display 3 sample sentences.

**(d)** Write all remaining sentences to a file named `nlp_ready_output.txt`, one sentence per
line. Re-read the file and print: (i) total lines written, (ii) the longest sentence by character
length, and (iii) the shortest sentence by character length.


In [9]:
# Task 3(a): Extract text and clean it step by step
import PyPDF2
import re

with open("sample.pdf", "rb") as f:
    reader = PyPDF2.PdfReader(f)
    full_text = ""
    for page in reader.pages:
        full_text += (page.extract_text() or "") + " "

print("Character count BEFORE cleaning:", len(full_text))

# (i) remove URLs
cleaned = re.sub(r"http\S+|www\.\S+", " ", full_text)
# (ii) remove digits
cleaned = re.sub(r"\d+", " ", cleaned)
# (iii) remove non-alphabetic characters except spaces and full stops
cleaned = re.sub(r"[^a-zA-Z\s.]", " ", cleaned)
# (iv) collapse multiple spaces into one
cleaned = re.sub(r"\s+", " ", cleaned).strip()

print("Character count AFTER cleaning:", len(cleaned))

Character count BEFORE cleaning: 3000
Character count AFTER cleaning: 2952


In [10]:
# Task 3(b): Lowercase, split into sentences, drop empties
cleaned_lower = cleaned.lower()

raw_sentences = re.split(r"[.!?]", cleaned_lower)
print("Sentence count BEFORE removing empty strings:", len(raw_sentences))

sentences = [s.strip() for s in raw_sentences]
sentences = [s for s in sentences if s != ""]
print("Sentence count AFTER removing empty strings:", len(sentences))

Sentence count BEFORE removing empty strings: 30
Sentence count AFTER removing empty strings: 29


In [11]:
# Task 3(c): Stopword removal + minimum-length filter using NLTK
%pip install nltk
import nltk

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    try:
        nltk.download("stopwords", quiet=True)
    except Exception:
        pass

try:
    from nltk.corpus import stopwords
    nltk_stopwords = set(stopwords.words("english"))
except LookupError:
    # Fallback: identical to NLTK's built-in English stopword list, used
    # only if the corpus cannot be downloaded (e.g. no internet access).
    nltk_stopwords = {
        'a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an',
        'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been',
        'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn',
        "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't",
        'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from',
        'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven',
        "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself',
        "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in',
        'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself',
        "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most',
        'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not',
        'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours',
        'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she',
        "she'd", "she'll", "she's", 'should', 'shouldn', "shouldn't", "should've",
        'so', 'some', 'such', 't', 'than', 'that', "that'll", 'the', 'their', 'theirs',
        'them', 'themselves', 'then', 'there', 'these', 'they', "they'd", "they'll",
        "they're", "they've", 'this', 'those', 'through', 'to', 'too', 'under',
        'until', 'up', 've', 'very', 'was', 'wasn', "wasn't", 'we', "we'd", "we'll",
        "we're", 'were', 'weren', "weren't", "we've", 'what', 'when', 'where',
        'which', 'while', 'who', 'whom', 'why', 'will', 'with', 'won', "won't",
        'wouldn', "wouldn't", 'y', 'you', "you'd", "you'll", 'your', "you're",
        'yours', 'yourself', 'yourselves', "you've"
    }

before_count = len(sentences)
filtered_sentences = []
for sentence in sentences:
    words = sentence.split()
    words_no_stop = [w for w in words if w not in nltk_stopwords]
    if len(words_no_stop) >= 5:
        filtered_sentences.append(sentence)

after_count = len(filtered_sentences)
print("Sentence count BEFORE stopword/length filter:", before_count)
print("Sentence count AFTER stopword/length filter:", after_count)

print("\nSample sentences:")
for s in filtered_sentences[:3]:
    print("-", s)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 3.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [nltk]4/5 [nltk]]

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Sentence count BEFORE stopword/length filter: 29
Sentence count AFTER stopword/length filter: 29

Sample sentences:
- page sample document artificial intelligence is transforming the way people work and live
- machine learning models can now process natural language recognize images and even generate creative content
- researchers around the world are studying the impact of artificial intelligence on society


In [12]:
# Task 3(d): Write remaining sentences to file, then re-read and summarize
with open("nlp_ready_output.txt", "w", encoding="utf-8") as f:
    for sentence in filtered_sentences:
        f.write(sentence + "\n")

with open("nlp_ready_output.txt", "r", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f.readlines()]

print("Total lines written:", len(lines))

if lines:
    longest = max(lines, key=len)
    shortest = min(lines, key=len)
    print("\nLongest sentence (", len(longest), "chars ):", longest)
    print("\nShortest sentence (", len(shortest), "chars ):", shortest)
else:
    print("No sentences were written to the file.")

Total lines written: 29

Longest sentence ( 160 chars ): the combination of data science techniques and domain expertise in medicine is opening new possibilities for personalized healthcare in the twenty first century

Shortest sentence ( 65 chars ): nutritionists recommend limiting processed foods and added sugars


---
## Task 4: DOCX Paragraph Classification and Statistical Analysis

**(a)** Open any `.docx` file using python-docx. For each paragraph, store its text and
`style.name` in a list of dictionaries. Print the total paragraph count and all unique style
names found in the document.

**(b)** Classify all paragraphs into exactly three groups: (i) Headings — style name starts with
"Heading", (ii) Body — style is "Normal" or contains "Body", (iii) Other/Empty — everything else
including blank paragraphs. Print the count of each group.

**(c)** For every Body paragraph with more than 5 words, compute its word count and character
count excluding spaces. Print: (i) the paragraph with the highest word count, (ii) the paragraph
with the lowest word count above 5 words, and (iii) the mean word count across all qualifying
body paragraphs rounded to 2 decimal places.

**(d)** Build a nested dictionary structured as
`{ style_name: { "count": N, "texts": [ list ] } }` for all styles found. Sort the outer
dictionary by count in descending order. If a style has more than 2 paragraphs, show only the
first 2 texts. Pretty-print the result.


In [ ]:
# Task 4(a): Read paragraphs and their styles
from docx import Document

doc = Document("docA.docx")

paragraphs_info = []
for para in doc.paragraphs:
    paragraphs_info.append({"text": para.text, "style": para.style.name})

print("Total paragraph count:", len(paragraphs_info))

unique_styles = sorted(set(p["style"] for p in paragraphs_info))
print("Unique style names found:")
for s in unique_styles:
    print("-", s)

NameError: name 'Document' is not defined

In [ ]:
# Task 4(b): Classify paragraphs into Headings / Body / Other-Empty
headings, body, other = [], [], []

for p in paragraphs_info:
    style = p["style"]
    text = p["text"].strip()
    if text == "":
        other.append(p)
    elif style.startswith("Heading"):
        headings.append(p)
    elif style == "Normal" or "Body" in style:
        body.append(p)
    else:
        other.append(p)

print("Headings:", len(headings))
print("Body:", len(body))
print("Other/Empty:", len(other))

In [ ]:
# Task 4(c): Word-count statistics for qualifying Body paragraphs (> 5 words)
qualifying = []
for p in body:
    words = p["text"].split()
    if len(words) > 5:
        qualifying.append({
            "text": p["text"],
            "word_count": len(words),
            "char_count": len(p["text"].replace(" ", ""))
        })

if qualifying:
    highest = max(qualifying, key=lambda x: x["word_count"])
    lowest = min(qualifying, key=lambda x: x["word_count"])
    mean_wc = round(sum(q["word_count"] for q in qualifying) / len(qualifying), 2)

    print(f"HIGHEST word count ({highest['word_count']} words):\n  {highest['text']}\n")
    print(f"LOWEST word count above 5 ({lowest['word_count']} words):\n  {lowest['text']}\n")
    print("Mean word count across qualifying body paragraphs:", mean_wc)
else:
    print("No body paragraphs with more than 5 words were found.")

In [ ]:
# Task 4(d): Nested style dictionary, sorted by count, pretty-printed
import json
from collections import defaultdict

style_dict = defaultdict(lambda: {"count": 0, "texts": []})

for p in paragraphs_info:
    style_dict[p["style"]]["count"] += 1
    style_dict[p["style"]]["texts"].append(p["text"])

sorted_styles = dict(sorted(style_dict.items(), key=lambda item: item[1]["count"], reverse=True))

final_dict = {}
for style, info in sorted_styles.items():
    texts = info["texts"][:2] if info["count"] > 2 else info["texts"]
    final_dict[style] = {"count": info["count"], "texts": texts}

print(json.dumps(final_dict, indent=2))

---
## Task 5: DOCX Table Extraction and Pandas Integration

**(a)** Open a `.docx` file that contains at least one table. Iterate through `doc.tables` and
extract each table as a list of lists: `[[row1col1, row1col2, ...], [row2col1, ...]]`. Print the
total number of tables found and the dimensions (rows × columns) of each.

**(b)** Take the first table's data, treat the first row as the header and the remaining rows as
data, and create a pandas DataFrame. Print the full DataFrame. Then count and print the number of
empty or None cells per column.

**(c)** Extract all paragraphs that appear before the first table in the document by iterating
through `doc.element.body` children and stopping once a `w:tbl` element is encountered. Store
these paragraphs in a list and print their count.

**(d)** Write both the pre-table paragraphs and the first table's data into a single file named
`docx_extracted.txt`. The file must have clearly labeled sections: a `--- PRE-TABLE PARAGRAPHS ---`
header followed by the paragraphs, then a `--- TABLE DATA ---` header followed by rows formatted
as pipe-separated values, e.g., `col1 | col2 | col3`.


In [ ]:
# Task 5(a): Extract every table as a list of lists
from docx import Document

doc = Document("docA.docx")  # this file contains one table (Team Members)

tables_data = []
for table in doc.tables:
    table_rows = []
    for row in table.rows:
        table_rows.append([cell.text for cell in row.cells])
    tables_data.append(table_rows)

print("Total number of tables found:", len(tables_data))
for i, t in enumerate(tables_data, start=1):
    rows = len(t)
    cols = len(t[0]) if rows > 0 else 0
    print(f"Table {i}: {rows} rows x {cols} columns")

In [ ]:
# Task 5(b): First table -> pandas DataFrame, then count empty/None cells per column
import pandas as pd

first_table = tables_data[0]
header = first_table[0]
data_rows = first_table[1:]

df = pd.DataFrame(data_rows, columns=header)
print(df)

empty_counts = df.apply(lambda col: col.apply(lambda x: x is None or str(x).strip() == "").sum())
print("\nEmpty/None cells per column:")
print(empty_counts)

In [ ]:
# Task 5(c): Paragraphs that appear before the first table
from docx.text.paragraph import Paragraph

pre_table_paragraphs = []
body_el = doc.element.body

for child in body_el.iterchildren():
    if child.tag.endswith("}tbl"):
        break
    if child.tag.endswith("}p"):
        para = Paragraph(child, doc)
        pre_table_paragraphs.append(para.text)

print("Number of paragraphs before the first table:", len(pre_table_paragraphs))
for t in pre_table_paragraphs:
    print("-", t)

In [ ]:
# Task 5(d): Write pre-table paragraphs + table data to docx_extracted.txt
with open("docx_extracted.txt", "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for para in pre_table_paragraphs:
        f.write(para + "\n")

    f.write("\n--- TABLE DATA ---\n")
    for row in first_table:
        f.write(" | ".join(row) + "\n")

print("docx_extracted.txt written successfully.\n")

with open("docx_extracted.txt", "r", encoding="utf-8") as f:
    print(f.read())

---
## Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

**(a)** Open two different `.docx` files (Document A and Document B). From each, extract all
paragraphs whose style name starts with "Heading" into two separate lists. Using Python set
operations, print: (i) headings common to both documents, (ii) headings only in Document A, and
(iii) headings only in Document B.

**(b)** From each document, extract all "Normal" style paragraphs. Merge both lists. Remove exact
duplicate paragraphs by comparing stripped, lowercased text. Print the total count before and
after deduplication.

**(c)** Tokenize every paragraph in the deduplicated list into words (lowercase, punctuation
removed via `re.sub()`). Build a combined word frequency dictionary using
`collections.Counter`. Remove at least 15 stopwords. Display the top 20 most frequent words.

**(d)** Compute and print a summary containing: (i) total headings across both documents, (ii)
total unique body paragraphs after deduplication, (iii) total unique word types (vocabulary
size), (iv) average paragraph length in words rounded to 2 decimal places, and (v) the single
longest paragraph by word count from the combined set.


In [ ]:
# Task 6(a): Compare headings between Document A and Document B
from docx import Document

doc_a = Document("docA.docx")
doc_b = Document("docB.docx")

headings_a = {p.text.strip() for p in doc_a.paragraphs
              if p.style.name.startswith("Heading") and p.text.strip() != ""}
headings_b = {p.text.strip() for p in doc_b.paragraphs
              if p.style.name.startswith("Heading") and p.text.strip() != ""}

common = headings_a & headings_b
only_a = headings_a - headings_b
only_b = headings_b - headings_a

print("Headings common to both documents:", common)
print("Headings only in Document A:", only_a)
print("Headings only in Document B:", only_b)

In [ ]:
# Task 6(b): Merge "Normal" paragraphs and remove duplicates
normal_a = [p.text.strip() for p in doc_a.paragraphs if p.style.name == "Normal" and p.text.strip() != ""]
normal_b = [p.text.strip() for p in doc_b.paragraphs if p.style.name == "Normal" and p.text.strip() != ""]

combined = normal_a + normal_b
print("Total paragraphs BEFORE deduplication:", len(combined))

seen = set()
deduplicated = []
for para in combined:
    key = para.strip().lower()
    if key not in seen:
        seen.add(key)
        deduplicated.append(para)

print("Total paragraphs AFTER deduplication:", len(deduplicated))

In [ ]:
# Task 6(c): Combined word frequency across deduplicated paragraphs
import re
from collections import Counter

stopwords_15 = {
    "the", "is", "in", "and", "to", "of", "a", "an", "for", "on",
    "with", "as", "are", "by", "this", "that", "it", "from", "at", "be"
}

all_words = []
for para in deduplicated:
    cleaned = re.sub(r"[^a-zA-Z\s]", " ", para).lower()
    all_words.extend(cleaned.split())

filtered_words = [w for w in all_words if w not in stopwords_15]
word_freq = Counter(filtered_words)

print("Top 20 most frequent words:")
for word, count in word_freq.most_common(20):
    print(f"{word}: {count}")

In [ ]:
# Task 6(d): Final summary across both documents
total_headings = len(headings_a) + len(headings_b)
total_unique_body = len(deduplicated)
vocab_size = len(word_freq)
avg_para_len = round(sum(len(p.split()) for p in deduplicated) / len(deduplicated), 2)
longest_para = max(deduplicated, key=lambda p: len(p.split()))

print("SUMMARY")
print("Total headings across both documents:", total_headings)
print("Total unique body paragraphs after deduplication:", total_unique_body)
print("Total unique word types (vocabulary size):", vocab_size)
print("Average paragraph length in words:", avg_para_len)
print("Longest paragraph by word count:")
print(" ", longest_para)

---
## Task 7: Multi-Endpoint API Data Joining and Analysis

**(a)** Send GET requests to both endpoints below. For each, verify the HTTP status code is
exactly 200; if not, raise a `ConnectionError` with the actual status code in the message. Parse
both responses as JSON lists.
- `https://jsonplaceholder.typicode.com/users`
- `https://jsonplaceholder.typicode.com/posts`

**(b)** Join the two datasets using `id` from users and `userId` from posts. Build a dictionary
structured as: `{ user_name: { "email": "...", "post_count": N, "titles": [ list of post titles ] } }`.
Print the result for the first 3 users only.

**(c)** From the joined dictionary, find the user who has written the most posts. Print their:
(i) full name, (ii) email address, (iii) company name, and (iv) all post titles numbered.

**(d)** Filter users where either of the following is true: (i) `company.bs` field contains the
word "synergies" (case-insensitive), or (ii) `address.city` has more than 8 characters. Print the
name, city, and post count for all matching users.

*(Requires internet access — see note at the top of the notebook.)*


In [ ]:
# Task 7(a): Fetch users and posts, validating the status code
import requests

users_url = "https://jsonplaceholder.typicode.com/users"
posts_url = "https://jsonplaceholder.typicode.com/posts"

resp_users = requests.get(users_url)
if resp_users.status_code != 200:
    raise ConnectionError(f"Failed to fetch users. Status code: {resp_users.status_code}")

resp_posts = requests.get(posts_url)
if resp_posts.status_code != 200:
    raise ConnectionError(f"Failed to fetch posts. Status code: {resp_posts.status_code}")

users = resp_users.json()
posts = resp_posts.json()

print("Users fetched:", len(users))
print("Posts fetched:", len(posts))

In [ ]:
# Task 7(b): Join users with their posts
user_posts = {}

for user in users:
    user_id = user["id"]
    user_name = user["name"]
    matching_posts = [p for p in posts if p["userId"] == user_id]
    user_posts[user_name] = {
        "email": user["email"],
        "post_count": len(matching_posts),
        "titles": [p["title"] for p in matching_posts]
    }

for i, (name, info) in enumerate(user_posts.items()):
    if i >= 3:
        break
    print(name, "->", info)

In [ ]:
# Task 7(c): User with the most posts
top_user_name = max(user_posts, key=lambda name: user_posts[name]["post_count"])
top_user_info = user_posts[top_user_name]
top_user_full = next(u for u in users if u["name"] == top_user_name)

print("Full name:", top_user_name)
print("Email:", top_user_info["email"])
print("Company name:", top_user_full["company"]["name"])
print("\nAll post titles:")
for i, title in enumerate(top_user_info["titles"], start=1):
    print(f"{i}. {title}")

In [ ]:
# Task 7(d): Filter users by company.bs containing "synergies" or long city names
matching_users = []
for user in users:
    bs = user["company"]["bs"].lower()
    city = user["address"]["city"]
    if "synergies" in bs or len(city) > 8:
        matching_users.append(user)

print("Users matching filter criteria:")
for u in matching_users:
    post_count = user_posts[u["name"]]["post_count"]
    print(f"Name: {u['name']} | City: {u['address']['city']} | Post count: {post_count}")

---
## Task 8: To-Do Completion Rate Analysis Across Users

**(a)** Fetch data from the endpoint below and parse it into a Python list. Validate that: (i)
the response status is 200, (ii) the parsed result is a list, (iii) the list contains at least
one record. If any validation fails, print a meaningful error and exit. Print the total number of
todo records retrieved.
- `https://jsonplaceholder.typicode.com/todos`

**(b)** Group todos by `userId`. For each user, compute and store:
`{ userId: { "total": N, "completed": N, "incomplete": N } }`. Print the result for all 10 users.
For each user, verify that `completed + incomplete == total` and print `True` or `False`
accordingly.

**(c)** Compute the completion rate for each user as `completed / total * 100` rounded to 1
decimal place. Print: (i) the userId with the highest completion rate and its value, (ii) the
userId with the lowest completion rate and its value, and (iii) the overall average completion
rate across all users.

**(d)** Fetch `https://jsonplaceholder.typicode.com/users` and build a lookup dictionary
`{ userId: user_name }`. Print a formatted table with columns:
`Rank | Name | Total Tasks | Completed | Completion Rate %`, sorted by completion rate
descending. Use f-strings or `str.ljust()` to align columns neatly.

*(Requires internet access — see note at the top of the notebook.)*


In [ ]:
# Task 8(a): Fetch and validate the todos list
import requests
import sys

todos_url = "https://jsonplaceholder.typicode.com/todos"
response = requests.get(todos_url)

if response.status_code != 200:
    print(f"Error: expected status 200 but got {response.status_code}")
    sys.exit(1)

todos = response.json()

if not isinstance(todos, list):
    print("Error: response is not a list")
    sys.exit(1)

if len(todos) == 0:
    print("Error: no todo records found")
    sys.exit(1)

print("Total todo records retrieved:", len(todos))

In [ ]:
# Task 8(b): Group todos by userId and verify totals
from collections import defaultdict

user_todo_stats = defaultdict(lambda: {"total": 0, "completed": 0, "incomplete": 0})

for todo in todos:
    uid = todo["userId"]
    user_todo_stats[uid]["total"] += 1
    if todo["completed"]:
        user_todo_stats[uid]["completed"] += 1
    else:
        user_todo_stats[uid]["incomplete"] += 1

for uid in sorted(user_todo_stats):
    stats = user_todo_stats[uid]
    check = (stats["completed"] + stats["incomplete"]) == stats["total"]
    print(f"User {uid}: {stats}  ->  completed+incomplete==total: {check}")

In [ ]:
# Task 8(c): Completion rate per user, highest/lowest/average
completion_rates = {}
for uid, stats in user_todo_stats.items():
    rate = round((stats["completed"] / stats["total"]) * 100, 1)
    completion_rates[uid] = rate

highest_uid = max(completion_rates, key=completion_rates.get)
lowest_uid = min(completion_rates, key=completion_rates.get)
overall_avg = round(sum(completion_rates.values()) / len(completion_rates), 1)

print(f"Highest completion rate: User {highest_uid} -> {completion_rates[highest_uid]}%")
print(f"Lowest completion rate: User {lowest_uid} -> {completion_rates[lowest_uid]}%")
print(f"Overall average completion rate: {overall_avg}%")

In [ ]:
# Task 8(d): Formatted, ranked table of completion rates with user names
users_resp = requests.get("https://jsonplaceholder.typicode.com/users")
if users_resp.status_code != 200:
    raise ConnectionError(f"Failed to fetch users. Status code: {users_resp.status_code}")

users = users_resp.json()
user_lookup = {u["id"]: u["name"] for u in users}

ranked = sorted(completion_rates.items(), key=lambda x: x[1], reverse=True)

print(f"{'Rank'.ljust(6)}{'Name'.ljust(20)}{'Total Tasks'.ljust(14)}{'Completed'.ljust(12)}{'Completion Rate %'}")
for rank, (uid, rate) in enumerate(ranked, start=1):
    name = user_lookup.get(uid, "Unknown")
    total = user_todo_stats[uid]["total"]
    completed = user_todo_stats[uid]["completed"]
    print(f"{str(rank).ljust(6)}{name.ljust(20)}{str(total).ljust(14)}{str(completed).ljust(12)}{rate}")

---
## Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

**(a)** Fetch data from `https://jsonplaceholder.typicode.com/users`. Each user contains nested
objects (`address` with `geo`, and `company`). Write a function `flatten_user(user: dict) -> dict`
that returns a single-level dictionary with exactly these keys: `id`, `name`, `username`, `email`,
`city`, `zipcode`, `lat`, `lng`, `company_name`, `company_catchphrase`. Apply this function to all
10 users and print the resulting flattened list.

**(b)** Create a pandas DataFrame from the flattened user list and display it in full. Then: (i)
cast the `lat` and `lng` columns to float, (ii) add a new column `distance_from_origin` computed
as `sqrt(lat² + lng²)` using the `math` module applied row-by-row, and (iii) print the user with
the greatest distance from origin.

**(c)** Using the `lat` column, classify users into two groups: (i) Northern Hemisphere where
`lat > 0`, and (ii) Southern Hemisphere where `lat < 0`. Print the names of users in each group.
Then determine which hemisphere has the higher average `distance_from_origin`.

**(d)** Fetch both endpoints below. For each post, count its number of comments by matching
`postId` in comments against `id` in posts. Build a sorted list of tuples:
`(post_id, post_title[:40], comment_count)` in descending order by comment count. Print the top 5
most-commented and bottom 5 least-commented posts.
- `https://jsonplaceholder.typicode.com/posts`
- `https://jsonplaceholder.typicode.com/comments`

*(Requires internet access — see note at the top of the notebook.)*


In [ ]:
# Task 9(a): Flatten nested user JSON into a single-level dictionary
import requests

resp = requests.get("https://jsonplaceholder.typicode.com/users")
if resp.status_code != 200:
    raise ConnectionError(f"Failed to fetch users. Status code: {resp.status_code}")
users = resp.json()

def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"]
    }

flattened_users = [flatten_user(u) for u in users]
for fu in flattened_users:
    print(fu)

In [ ]:
# Task 9(b): DataFrame + distance_from_origin using math.sqrt
import pandas as pd
import math

df = pd.DataFrame(flattened_users)
pd.set_option("display.max_columns", None)
print(df)

df["lat"] = df["lat"].astype(float)
df["lng"] = df["lng"].astype(float)

df["distance_from_origin"] = df.apply(lambda row: math.sqrt(row["lat"]**2 + row["lng"]**2), axis=1)

farthest_user = df.loc[df["distance_from_origin"].idxmax()]
print("\nUser with greatest distance from origin:")
print(farthest_user)

In [ ]:
# Task 9(c): Hemisphere classification and average distance comparison
northern = df[df["lat"] > 0]
southern = df[df["lat"] < 0]

print("Northern Hemisphere users:", list(northern["name"]))
print("Southern Hemisphere users:", list(southern["name"]))

north_avg = northern["distance_from_origin"].mean()
south_avg = southern["distance_from_origin"].mean()

print(f"\nNorthern hemisphere average distance: {north_avg:.2f}")
print(f"Southern hemisphere average distance: {south_avg:.2f}")

if north_avg > south_avg:
    print("Northern Hemisphere has the higher average distance from origin.")
else:
    print("Southern Hemisphere has the higher average distance from origin.")

In [ ]:
# Task 9(d): Comment counts per post, top 5 / bottom 5
posts_resp = requests.get("https://jsonplaceholder.typicode.com/posts")
comments_resp = requests.get("https://jsonplaceholder.typicode.com/comments")

if posts_resp.status_code != 200 or comments_resp.status_code != 200:
    raise ConnectionError("Failed to fetch posts or comments")

posts = posts_resp.json()
comments = comments_resp.json()

comment_counts = {}
for post in posts:
    pid = post["id"]
    comment_counts[pid] = sum(1 for c in comments if c["postId"] == pid)

post_comment_list = [(p["id"], p["title"][:40], comment_counts[p["id"]]) for p in posts]
post_comment_list.sort(key=lambda x: x[2], reverse=True)

print("Top 5 most-commented posts:")
for pid, title, count in post_comment_list[:5]:
    print(f"Post {pid}: '{title}' -> {count} comments")

print("\nBottom 5 least-commented posts:")
for pid, title, count in post_comment_list[-5:]:
    print(f"Post {pid}: '{title}' -> {count} comments")